In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
%cd ..

/path/to/repo/my-coles-gnn-experimetns/scenario_gender


/path/to/repo/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [7]:
import sys; import os; sys.path.append(os.path.dirname(os.getcwd()))

In [8]:
from typing import List

from omegaconf import OmegaConf 

from ptls_extension_2024_research.latex_table_creation.latex_table_creation import create_latex_table, get_metrics, get_experiment_dicts_list
from ptls_extension_2024_research.latex_table_creation.experiment_dicts_list_modifiers import bolden_top_k, sort_by_col
from ptls_extension_2024_research.latex_table_creation.prefix_map import get_idxs_where_all_metrics_superpass, prefix_map_from_idx_lst
from ptls_extension_2024_research.latex_table_creation.hyperparam_getters import get_batchsize_from_config, get_split_count_from_config, get_triplets_per_user_from_config, get_convex_loss_alpha_from_config, get_min_users_in_separated_single_bin_from_config


/path/to/repo/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
COLES_METRICS = {r"\textbf{AUC test}": 0.877, r"\textbf{ACC test}": 0.793}

In [10]:
HYDRA_OUTPUTS_PATH ='hydra_outputs/'

In [11]:
METRIC_REPORT_NAME_TO_METRIC_TABLE_NAME = {'auroc': r"\textbf{AUC test}", 'accuracy': r"\textbf{ACC test}"}

In [12]:
REPORT_FILE_PATH = "results/scenario_gender_concat_new.txt"

In [13]:
CONCAT_FEATS_PREFIX_STR_OPTIONS = ['gr_merge-', 'gr_unweighted-', 'gr_weighted-']

In [14]:
def leave_only_experiments_from_results(expected_experiments: List[str], report_file_path) -> List[str]:
    with open(report_file_path, 'r') as f:
        report_file_content = f.read()
    
    existing_experiments = [exp_name for exp_name in expected_experiments if exp_name in report_file_content]
    return existing_experiments

In [15]:
def get_raw_concatenated_features_from_name(config, experiment_name: str) -> str:
    for concat_feats_prefix_str in CONCAT_FEATS_PREFIX_STR_OPTIONS:
        if experiment_name.startswith(concat_feats_prefix_str):
            return concat_feats_prefix_str
    return None

def get_preprocessed_concatenated_features_from_name(config, experiment_name: str) -> str:
    concat_feats_prefix_str = get_raw_concatenated_features_from_name(config, experiment_name)
    if concat_feats_prefix_str is None:
        return "N/A"
    return concat_feats_prefix_str.replace('_', ' ').replace('-', '')

def remove_concat_prefix_str(experiment_name: str) -> str:
    concat_feats_prefix_str = get_raw_concatenated_features_from_name(None, experiment_name)
    if concat_feats_prefix_str is None:
        return experiment_name
    return experiment_name[len(concat_feats_prefix_str) :]

In [16]:
all_experiments_dicts_list = []

# BPR

In [17]:
PRETRAINED_MODELS_ROOT = 'models/pretrained_gnn_models'

In [18]:
# report_file_path="results/scenario_gender_bpr.txt"

In [19]:
MAX_EPOCHES=40
N_TRIPLETS_PER_ANCHOR_USER_OPTIONS=(128, 32, 8, 1)
MIN_ELEMENTS_IN_BIN_WHEN_ONE_BIN_ONLY_OPTIONS=(5,)
LOSS_ALPHA_OPTIONS=(0.5, 0.85, 0.15, 0.01, 0)


expected_experiments = []

for min_elements_in_bin_when_one_bin_only in MIN_ELEMENTS_IN_BIN_WHEN_ONE_BIN_ONLY_OPTIONS:
    for n_triplets_per_anchor_user in N_TRIPLETS_PER_ANCHOR_USER_OPTIONS:
        for loss_alpha in LOSS_ALPHA_OPTIONS:
            for concat_feats_prefix_str in CONCAT_FEATS_PREFIX_STR_OPTIONS:
                experiment_name = f'{concat_feats_prefix_str}coles_bpr__loss_alpha_{loss_alpha}__{MAX_EPOCHES}__epoches__triplets_per_user_{n_triplets_per_anchor_user}__bin_separation_margin_{min_elements_in_bin_when_one_bin_only}__try_2'
                expected_experiments.append(experiment_name)



In [20]:
len(expected_experiments)

60

In [21]:
expected_experiments = leave_only_experiments_from_results(expected_experiments, REPORT_FILE_PATH)

In [22]:
len(expected_experiments)

6

In [23]:
hyperparams_to_getters = {
    r"\textbf{Triplets per user}": get_triplets_per_user_from_config,
    r"\textbf{Loss alpha}": get_convex_loss_alpha_from_config,
    r"\textbf{Min users in separated single bin}": get_min_users_in_separated_single_bin_from_config,
    r"\textbf{Concatenated features}": get_preprocessed_concatenated_features_from_name,
}

In [24]:
experiment_dicts_list = get_experiment_dicts_list(
    expected_experiments, hyperparams_to_getters, HYDRA_OUTPUTS_PATH, 
    experiment_name_to_main_experiment_name = remove_concat_prefix_str, 
    report_file_path = REPORT_FILE_PATH,
    metric_report_name_to_metric_table_name = METRIC_REPORT_NAME_TO_METRIC_TABLE_NAME)

In [25]:
all_experiments_dicts_list.extend([{k: v for k, v in d.items()} for d in experiment_dicts_list])

In [26]:
WEIGHTS_TYPE = 'type 2'


experiment_dicts_list = sort_by_col(experiment_dicts_list, r"\textbf{AUC test}")
prefix_map = prefix_map_from_idx_lst(get_idxs_where_all_metrics_superpass(experiment_dicts_list, COLES_METRICS), "\n\\rowcolor{gray!50}\n")
experiment_dicts_list = bolden_top_k(experiment_dicts_list, 3, [r"\textbf{AUC test}", r"\textbf{ACC test}"])



EXPERIMENT_NAME = r'\makecell{Coles only with bpr}'

experiment_data = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: experiment_dicts_list
    }
}



In [27]:
experiment_dicts_list[0].keys()

dict_keys(['\\textbf{Triplets per user}', '\\textbf{Loss alpha}', '\\textbf{Min users in separated single bin}', '\\textbf{Concatenated features}', '\\textbf{AUC test}', '\\textbf{ACC test}'])

In [28]:
hyperparameters = hyperparameter_header_strs = list(experiment_dicts_list[0].keys())

caption = "Comparison of different setups where coles was trained with contrastive loss alongside with bpr loss. Bpr loss chooses positive and negative users according to similarity of user embeddings extracted from adjacency matrix"

row_prefix_dict = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: prefix_map
    }
}

latex_table = create_latex_table(experiment_data, hyperparameters, hyperparameter_header_strs, caption, row_prefix_dict)

print(latex_table)

\begin{table*}[h!]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|c|c|}
\hline
\textbf{Model} & \textbf{Type of weights} & \textbf{Triplets per user} & \textbf{Loss alpha} & \textbf{Min users in separated single bin} & \textbf{Concatenated features} & \textbf{AUC test} & \textbf{ACC test}\\
\hline

\rowcolor{gray!50}
\multirow{6}{*}{\makecell{Coles only with bpr}} &\multirow{6}{*}{type 2} & 128 & 0.85 & 5 & gr merge & \textbf{0.887} & \textbf{0.801} \\ \cline{3-8} 

\rowcolor{gray!50}
& &128 & 0.85 & 5 & gr weighted & \textbf{0.887} & 0.796 \\ \cline{3-8} 

\rowcolor{gray!50}
& &128 & 0.15 & 5 & gr weighted & \textbf{0.887} & \textbf{0.802} \\ \cline{3-8} 

\rowcolor{gray!50}
& &128 & 0.15 & 5 & gr merge & 0.886 & \textbf{0.804} \\ \cline{3-8} 
& &128 & 0.85 & 5 & gr unweighted & 0.884 & 0.792 \\ \cline{3-8} 

\rowcolor{gray!50}
& &128 & 0.15 & 5 & gr unweighted & 0.884 & 0.798 \\ \cline{2-8} 
\hline
\end{tabular}
}
\caption{Comparison of different setups where col

In [29]:
for d in all_experiments_dicts_list:
    print(type(d[r"\textbf{AUC test}"]))
    print(d[r"\textbf{AUC test}"])
    if type(d[r"\textbf{AUC test}"] == str):
        d

<class 'float'>
0.887
<class 'float'>
0.884
<class 'float'>
0.887
<class 'float'>
0.886
<class 'float'>
0.884
<class 'float'>
0.887


# BPR PRETRAINED

In [30]:
from collections import defaultdict
import os

CHECKPOINTS_DIR = "../../checkpoints"

dir_to_epoches = {
    "wl-0.5_gnn-GAT_res-True_wd-0": [15, 50, 75, 100, 150, 200],
    "wl-0.5_gnn-GAT_res-True_wd-0.1": [25, 50, 75, 100, 150, 200],
    "wl-0.5_gnn-GAT_res-True_wd-0.01": [25, 50, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-False_wd-0": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0": [75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.1": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.01": [25, 50, 75, 100, 150, 200],
}



MAX_EPOCHES=40
N_TRIPLETS_PER_ANCHOR_USER_OPTIONS=(128, 32, 8, 1)
MIN_ELEMENTS_IN_BIN_WHEN_ONE_BIN_ONLY_OPTIONS=(5,)
LOSS_ALPHA_OPTIONS=(0.5, 0.85, 0.15)


expected_experiments = []

for pretrained_gnn_name, pretrain_epoches in dir_to_epoches.items():
    for pretrain_epoch in pretrain_epoches:
        for min_elements_in_bin_when_one_bin_only in MIN_ELEMENTS_IN_BIN_WHEN_ONE_BIN_ONLY_OPTIONS:
            for n_triplets_per_anchor_user in N_TRIPLETS_PER_ANCHOR_USER_OPTIONS:
                for loss_alpha in LOSS_ALPHA_OPTIONS:
                    for concat_feats_prefix_str in CONCAT_FEATS_PREFIX_STR_OPTIONS:

                        experiment_name = f'{concat_feats_prefix_str}check__coles_bpr_with_pretrained_gnn__GNN_{pretrained_gnn_name}__pretrain_epoches_{pretrain_epoch}__loss_alpha_{loss_alpha}__triplets_per_user_{n_triplets_per_anchor_user}__bin_separation_margin_{min_elements_in_bin_when_one_bin_only}__{MAX_EPOCHES}__epoches__try_1'
                        ckpt_dir_path = os.path.join(CHECKPOINTS_DIR, experiment_name)
                        expected_experiments.append(experiment_name)

In [31]:
print(len(expected_experiments))

1332


In [32]:
expected_experiments = leave_only_experiments_from_results(expected_experiments, REPORT_FILE_PATH)

In [33]:
print(len(expected_experiments))

6


In [34]:
# hyperparams_to_getters = {
#     r"\textbf{Triplets per user}": lambda config: config['pl_module']['loss']['loss2']['triplet_selector']['num_triplets_per_anchor_user'],
#     r"\textbf{Loss alpha}": lambda config: config['pl_module']['loss']['alpha'],
#     r"\textbf{Min users in separated single bin}": lambda config: config['pl_module']['loss']['loss2']['triplet_selector']['bin_separation_strategy']['min_elements_in_bin']
# }

In [35]:
import re

hyperparams_to_getters = {
    r"\textbf{GNN}": lambda config, experiment_name: re.search(r'gnn-(GraphSAGE|GAT)', experiment_name).group(1),
    r"\textbf{Has residual connection}": lambda config, experiment_name: True if re.search(r'res-(True|False)', experiment_name).group(1) == "True" else False,
    r"\textbf{Triplets per user}": lambda config, experiment_name: int(re.search(r'triplets_per_user_(\d+)', experiment_name).group(1)),
    r"\textbf{Loss alpha}": lambda config, experiment_name: float(re.search(r'loss_alpha_(\d+\.\d+)', experiment_name).group(1)),
    r"\textbf{Min users in separated single bin}": lambda config, experiment_name: int(re.search(r'bin_separation_margin_(\d+)', experiment_name).group(1)),
    r"\textbf{GNN Loss alpha}": lambda config, experiment_name: float(re.search(r'wl-(\d+\.\d+)', experiment_name).group(1)),
    r"\textbf{Pretrain epoches}": lambda config, experiment_name: int(re.search(r'pretrain_epoches_(\d+)', experiment_name).group(1)),
    r"\textbf{Train epoches}": lambda config, experiment_name: int(re.search(r'(\d+)__epoches', experiment_name).group(1)),
    r"\textbf{Concatenated features}": get_preprocessed_concatenated_features_from_name,
}


In [36]:
experiment_dicts_list = get_experiment_dicts_list(
    expected_experiments, hyperparams_to_getters, HYDRA_OUTPUTS_PATH, 
    experiment_name_to_main_experiment_name = remove_concat_prefix_str, 
    report_file_path = REPORT_FILE_PATH,
    metric_report_name_to_metric_table_name = METRIC_REPORT_NAME_TO_METRIC_TABLE_NAME)

Warning! Config does not exists: hydra_outputs/check__coles_bpr_with_pretrained_gnn__GNN_wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_75__loss_alpha_0.85__triplets_per_user_128__bin_separation_margin_5__40__epoches__try_1/.hydra/config.yaml
Warning! Config does not exists: hydra_outputs/check__coles_bpr_with_pretrained_gnn__GNN_wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_75__loss_alpha_0.85__triplets_per_user_128__bin_separation_margin_5__40__epoches__try_1/.hydra/config.yaml
Warning! Config does not exists: hydra_outputs/check__coles_bpr_with_pretrained_gnn__GNN_wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_75__loss_alpha_0.85__triplets_per_user_128__bin_separation_margin_5__40__epoches__try_1/.hydra/config.yaml
Warning! Config does not exists: hydra_outputs/check__coles_bpr_with_pretrained_gnn__GNN_wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__loss_alpha_0.15__triplets_per_user_128__bin_separation_margin_5__40__epoches__try_1/.hydra/config.yaml
Warning! Config does 

In [37]:
all_experiments_dicts_list.extend([{k: v for k, v in d.items()} for d in experiment_dicts_list])

In [38]:
WEIGHTS_TYPE = 'type 2'


experiment_dicts_list = sort_by_col(experiment_dicts_list, r"\textbf{AUC test}")
prefix_map = prefix_map_from_idx_lst(get_idxs_where_all_metrics_superpass(experiment_dicts_list, COLES_METRICS), "\n\\rowcolor{gray!50}\n")
experiment_dicts_list = bolden_top_k(experiment_dicts_list, 3, [r"\textbf{AUC test}", r"\textbf{ACC test}"])




EXPERIMENT_NAME = r'\makecell{Coles + pretrained gnn with bpr}'

experiment_data = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: experiment_dicts_list
    }
}



In [39]:
hyperparameters = hyperparameter_header_strs = list(experiment_dicts_list[0].keys())

caption = r"Comparison of different setups where coles had item\_id (mcc_code) embedding layer initialized with pretrained gnn embeddings and was trained with contrastive loss alongside with bpr loss. Bpr loss chooses positive and negative users according to similarity of user embeddings extracted from adjacency matrix"

row_prefix_dict = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: prefix_map
    }
}

latex_table = create_latex_table(experiment_data, hyperparameters, hyperparameter_header_strs, caption, row_prefix_dict)

print(latex_table)

\begin{table*}[h!]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|c|c|c|c|c|c|c|}
\hline
\textbf{Model} & \textbf{Type of weights} & \textbf{GNN} & \textbf{Has residual connection} & \textbf{Triplets per user} & \textbf{Loss alpha} & \textbf{Min users in separated single bin} & \textbf{GNN Loss alpha} & \textbf{Pretrain epoches} & \textbf{Train epoches} & \textbf{Concatenated features} & \textbf{AUC test} & \textbf{ACC test}\\
\hline

\rowcolor{gray!50}
\multirow{6}{*}{\makecell{Coles + pretrained gnn with bpr}} &\multirow{6}{*}{type 2} & GraphSAGE & True & 128 & 0.15 & 5 & 0.5 & 100 & 40 & gr merge & \textbf{0.886} & \textbf{0.808} \\ \cline{3-13} 

\rowcolor{gray!50}
& &GAT & True & 128 & 0.85 & 5 & 0.5 & 75 & 40 & gr merge & \textbf{0.885} & 0.804 \\ \cline{3-13} 

\rowcolor{gray!50}
& &GAT & True & 128 & 0.85 & 5 & 0.5 & 75 & 40 & gr weighted & \textbf{0.885} & 0.8 \\ \cline{3-13} 

\rowcolor{gray!50}
& &GraphSAGE & True & 128 & 0.15 & 5 & 0.5 & 100 & 40 & gr w

# PRETRAINED

In [40]:
expected_experiment_names_no_concat = [
    "coles_gnn__pretrained_wl-0.5_gnn-GAT_res-True_wd-0.1__pretrain_epoches_75__epoches_40",
    "coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-False_wd-0__pretrain_epoches_75__epoches_40",
    "coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_150__ckpt_epoch_29",
    # "coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_40",  # Правильным является именно 
    "coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_40_embeddings__.pickle",
]

In [41]:
expected_experiments = [
    concat_feats_prefix_str+expected_experiment_name_no_concat
    for expected_experiment_name_no_concat in expected_experiment_names_no_concat
    for concat_feats_prefix_str in CONCAT_FEATS_PREFIX_STR_OPTIONS
]

In [42]:
len(expected_experiments)

12

In [43]:
with open(REPORT_FILE_PATH, 'r') as f:
    report_file_content = f.read()

In [44]:
for expected_experiment_name in expected_experiments:
    assert expected_experiment_name in report_file_content

AssertionError: 

In [45]:
def get_train_epochs_from_name(config, experiment_name: str) -> int:
    return int(re.search(r'__epoches_(\d+)', experiment_name).group(1))

hyperparams_to_getters = {
    r"\textbf{GNN}": lambda config, experiment_name: re.search(r'gnn-(GraphSAGE|GAT)', experiment_name).group(1),
    r"\textbf{Has residual connection}": lambda config, experiment_name: True if re.search(r'res-(True|False)', experiment_name).group(1) == "True" else False,
    r"\textbf{GNN Loss alpha}": lambda config, experiment_name: float(re.search(r'wl-(\d+\.\d+)', experiment_name).group(1)),
    r"\textbf{Pretrain epoches}": lambda config, experiment_name: int(re.search(r'pretrain_epoches_(\d+)', experiment_name).group(1)),
    r"\textbf{Train epoches}": get_train_epochs_from_name,
    r"\textbf{Concatenated features}": get_preprocessed_concatenated_features_from_name,
}

In [46]:
experiment_dicts_list = get_experiment_dicts_list(
    expected_experiments, hyperparams_to_getters, HYDRA_OUTPUTS_PATH, 
    experiment_name_to_main_experiment_name = remove_concat_prefix_str, 
    report_file_path = REPORT_FILE_PATH,
    metric_report_name_to_metric_table_name = METRIC_REPORT_NAME_TO_METRIC_TABLE_NAME)

Warning! Config does not exists: hydra_outputs/coles_gnn__pretrained_wl-0.5_gnn-GAT_res-True_wd-0.1__pretrain_epoches_75__epoches_40/.hydra/config.yaml
Could not find experiment gr_merge-coles_gnn__pretrained_wl-0.5_gnn-GAT_res-True_wd-0.1__pretrain_epoches_75__epoches_40 in file content
Warning! Config does not exists: hydra_outputs/coles_gnn__pretrained_wl-0.5_gnn-GAT_res-True_wd-0.1__pretrain_epoches_75__epoches_40/.hydra/config.yaml
Could not find experiment gr_unweighted-coles_gnn__pretrained_wl-0.5_gnn-GAT_res-True_wd-0.1__pretrain_epoches_75__epoches_40 in file content
Warning! Config does not exists: hydra_outputs/coles_gnn__pretrained_wl-0.5_gnn-GAT_res-True_wd-0.1__pretrain_epoches_75__epoches_40/.hydra/config.yaml
Could not find experiment gr_weighted-coles_gnn__pretrained_wl-0.5_gnn-GAT_res-True_wd-0.1__pretrain_epoches_75__epoches_40 in file content
Warning! Config does not exists: hydra_outputs/coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-False_wd-0__pretrain_epoches_75

In [47]:
all_experiments_dicts_list.extend([{k: v for k, v in d.items()} for d in experiment_dicts_list])

In [48]:
WEIGHTS_TYPE = 'type 2'


experiment_dicts_list = sort_by_col(experiment_dicts_list, r"\textbf{AUC test}")
prefix_map = prefix_map_from_idx_lst(get_idxs_where_all_metrics_superpass(experiment_dicts_list, COLES_METRICS), "\n\\rowcolor{gray!50}\n")
experiment_dicts_list = bolden_top_k(experiment_dicts_list, 3, [r"\textbf{AUC test}", r"\textbf{ACC test}"])




EXPERIMENT_NAME = r'\makecell{Coles + pretrained gnn with bpr}'

experiment_data = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: experiment_dicts_list
    }
}



In [49]:
hyperparameters = hyperparameter_header_strs = list(experiment_dicts_list[0].keys())

caption = r"Comparison of different setups where coles had item\_id (mcc_code) embedding layer initialized with pretrained gnn embeddings and was trained with contrastive loss alongside with bpr loss. Bpr loss chooses positive and negative users according to similarity of user embeddings extracted from adjacency matrix"

row_prefix_dict = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: prefix_map
    }
}

latex_table = create_latex_table(experiment_data, hyperparameters, hyperparameter_header_strs, caption, row_prefix_dict)

print(latex_table)

\begin{table*}[h!]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|c|c|c|c|}
\hline
\textbf{Model} & \textbf{Type of weights} & \textbf{GNN} & \textbf{Has residual connection} & \textbf{GNN Loss alpha} & \textbf{Pretrain epoches} & \textbf{Train epoches} & \textbf{Concatenated features} & \textbf{AUC test} & \textbf{ACC test}\\
\hline

\rowcolor{gray!50}
\multirow{3}{*}{\makecell{Coles + pretrained gnn with bpr}} &\multirow{3}{*}{type 2} & GraphSAGE & False & 0.5 & 75 & 40 & gr merge & \textbf{0.889} & \textbf{0.809} \\ \cline{3-10} 

\rowcolor{gray!50}
& &GraphSAGE & False & 0.5 & 75 & 40 & gr weighted & \textbf{0.889} & \textbf{0.811} \\ \cline{3-10} 

\rowcolor{gray!50}
& &GraphSAGE & False & 0.5 & 75 & 40 & gr unweighted & \textbf{0.887} & \textbf{0.81} \\ \cline{2-10} 
\hline
\end{tabular}
}
\caption{Comparison of different setups where coles had item\_id (mcc_code) embedding layer initialized with pretrained gnn embeddings and was trained with contrastive loss

# All experiments table

In [50]:
from typing import Dict, Any, Iterable


def get_all_hyperparam_keys_from_dicts_list(experiments_dicts_list: List[Dict[str, Any]]) -> Iterable[str]:
    hyperparam_keys = set()
    for experiment_dict in experiments_dicts_list:
        hyperparam_keys.update(experiment_dict.keys())
    return hyperparam_keys

def set_all_hyperparams_to_dicts(experiments_dicts_list: List[Dict[str, Any]], 
                                 all_hyperparam_keys: Iterable[str]
                                 ) -> List[Dict[str, Any]]:
    new_experiments_dicts_list = []
    for experiment_dict in experiments_dicts_list:
        new_dict = {k: "N/A" if k not in experiment_dict else experiment_dict[k] for k in all_hyperparam_keys}
        new_experiments_dicts_list.append(new_dict)
    return new_experiments_dicts_list

In [51]:
all_hyperparam_keys = get_all_hyperparam_keys_from_dicts_list(all_experiments_dicts_list)

In [52]:
all_experiments_dicts_list_with_all_keys = set_all_hyperparams_to_dicts(all_experiments_dicts_list, all_hyperparam_keys)

In [53]:
WEIGHTS_TYPE = 'type 2'


all_experiments_dicts_list_with_all_keys = sort_by_col(all_experiments_dicts_list_with_all_keys, r"\textbf{AUC test}")
prefix_map = prefix_map_from_idx_lst(get_idxs_where_all_metrics_superpass(all_experiments_dicts_list_with_all_keys, COLES_METRICS), "\n\\rowcolor{gray!50}\n")
all_experiments_dicts_list_with_all_keys = bolden_top_k(all_experiments_dicts_list_with_all_keys, 3, [r"\textbf{AUC test}", r"\textbf{ACC test}"])




EXPERIMENT_NAME = r'\makecell{Coles + pretrained gnn with bpr}'

experiment_data = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: all_experiments_dicts_list_with_all_keys
    }
}



In [54]:
hyperparameters = hyperparameter_header_strs = list(all_experiments_dicts_list_with_all_keys[0].keys())

caption = r"Comparison of different setups where coles had item\_id (mcc_code) embedding layer initialized with pretrained gnn embeddings and was trained with contrastive loss alongside with bpr loss. Bpr loss chooses positive and negative users according to similarity of user embeddings extracted from adjacency matrix"

row_prefix_dict = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: prefix_map
    }
}

latex_table = create_latex_table(experiment_data, hyperparameters, hyperparameter_header_strs, caption, row_prefix_dict)

print(latex_table)

\begin{table*}[h!]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|c|c|c|c|c|c|c|}
\hline
\textbf{Model} & \textbf{Type of weights} & \textbf{GNN} & \textbf{Pretrain epoches} & \textbf{Loss alpha} & \textbf{GNN Loss alpha} & \textbf{Min users in separated single bin} & \textbf{Concatenated features} & \textbf{AUC test} & \textbf{ACC test} & \textbf{Has residual connection} & \textbf{Train epoches} & \textbf{Triplets per user}\\
\hline

\rowcolor{gray!50}
\multirow{15}{*}{\makecell{Coles + pretrained gnn with bpr}} &\multirow{15}{*}{type 2} & GraphSAGE & 75 & N/A & 0.5 & N/A & gr merge & \textbf{0.889} & \textbf{0.809} & False & 40 & N/A \\ \cline{3-13} 

\rowcolor{gray!50}
& &GraphSAGE & 75 & N/A & 0.5 & N/A & gr weighted & \textbf{0.889} & \textbf{0.811} & False & 40 & N/A \\ \cline{3-13} 

\rowcolor{gray!50}
& &N/A & N/A & 0.85 & N/A & 5 & gr merge & \textbf{0.887} & 0.801 & N/A & N/A & 128 \\ \cline{3-13} 

\rowcolor{gray!50}
& &N/A & N/A & 0.85 & N/A & 5 & gr we